# Parcel ↔ Image Matching (cleaned final notebook)

This notebook rebuilds the pipeline from scratch with cleaner logic and a few accuracy improvements:

1. Extract image GPS + heading from EXIF  
2. Build projected camera points and triangular view cones  
3. Load parcel polygons in a projected CRS  
4. Generate candidate matches via spatial join  
5. Score candidates using **boundary distance + heading alignment**  
6. Select **one best parcel per image**  
7. Aggregate matched images **by parcel** for export and Folium visualization  

### Important logic choices
- **Best match** is selected **per image**
- **Folium markers** are plotted **per parcel**, so marker count can be smaller than image count
- Distance is computed to the **parcel boundary/polygon**, not only to the centroid
- Heading is used in the final score, not only in cone generation

In [1]:
from pathlib import Path
import os
import math
from html import escape

import numpy as np
import pandas as pd
import geopandas as gpd
import folium

from PIL import Image
from PIL.ExifTags import TAGS, GPSTAGS
from shapely.geometry import Point, Polygon
from shapely.ops import nearest_points

# -----------------------------
# Paths and core parameters
# -----------------------------
IMAGE_FOLDER = Path("Data/testImages")  # Original image folder
PARCEL_FILE  = Path("Data/Real_Property_Information.geojson")  # Baltimore parcel dataset

CRS_WGS84 = "EPSG:4326"
CRS_PROJ  = "EPSG:2248"

FOV_DEG = 50
CONE_LENGTH_FT = 130
MAX_BOUNDARY_DISTANCE_FT = 150
ANGLE_WEIGHT_FT_PER_DEG = 1.0
output_dir = Path("outputs")
IMAGE_METADATA_CSV = Path("image_metadata_extracted.csv")
BEST_MATCH_CSV = Path("image_best_match.csv")
PARCEL_MATCH_CSV = Path("parcel_image_join.csv")
PARCEL_MATCH_GEOJSON = Path("parcel_image_join.geojson")
FOLIUM_HTML = Path("parcel_map.html")

print("Image folder exists:", IMAGE_FOLDER.exists(), IMAGE_FOLDER)
print("Parcel file exists :", PARCEL_FILE.exists(), PARCEL_FILE)

Image folder exists: True Data/testImages
Parcel file exists : True Data/Real_Property_Information.geojson


In [2]:
def get_exif_data(image_path):
    try:
        img = Image.open(image_path)
        exif = img._getexif()
        if not exif:
            return {}
        return {TAGS.get(tag, tag): value for tag, value in exif.items()}
    except Exception:
        return {}

def get_gps_info(exif_data):
    gps = exif_data.get("GPSInfo")
    if not gps:
        return None
    return {GPSTAGS.get(k, k): v for k, v in gps.items()}

def _rational_to_float(x):
    try:
        if hasattr(x, "numerator") and hasattr(x, "denominator"):
            return float(x.numerator) / float(x.denominator)
        if isinstance(x, tuple) and len(x) == 2 and all(isinstance(v, (int, float)) for v in x):
            num, den = x
            return float(num) / float(den)
        return float(x)
    except Exception:
        return float("nan")

def dms_to_decimal(dms):
    try:
        d, m, s = dms
        d = _rational_to_float(d)
        m = _rational_to_float(m)
        s = _rational_to_float(s)
        return d + m / 60.0 + s / 3600.0
    except Exception:
        return None

def extract_lat_lon_heading(image_path):
    exif_data = get_exif_data(image_path)
    gps_info = get_gps_info(exif_data)
    if not gps_info:
        return None, None, None

    lat = dms_to_decimal(gps_info.get("GPSLatitude"))
    lon = dms_to_decimal(gps_info.get("GPSLongitude"))
    if lat is None or lon is None:
        return None, None, None

    if gps_info.get("GPSLatitudeRef", "N") != "N":
        lat = -lat
    if gps_info.get("GPSLongitudeRef", "E") != "E":
        lon = -lon

    heading = gps_info.get("GPSImgDirection", None)
    try:
        heading = _rational_to_float(heading) if heading is not None else None
        if pd.notna(heading):
            heading = float(heading) % 360
        else:
            heading = None
    except Exception:
        heading = None

    return lat, lon, heading

In [3]:
rows = []

for fname in sorted(os.listdir(IMAGE_FOLDER)):
    if fname.lower().endswith(".jpg"):
        path = IMAGE_FOLDER / fname
        lat, lon, heading = extract_lat_lon_heading(path)
        rows.append(
            {
                "filename": fname,
                "image_path": str(path.resolve()),
                "latitude": lat,
                "longitude": lon,
                "heading": heading,
            }
        )

df_imgs = pd.DataFrame(rows)
df_imgs.to_csv(IMAGE_METADATA_CSV, index=False)

print("Raw image rows:", len(df_imgs))
print("With lat/lon   :", df_imgs[["latitude", "longitude"]].dropna().shape[0])
print("With heading   :", df_imgs["heading"].notna().sum())
display(df_imgs.head())

Raw image rows: 578
With lat/lon   : 578
With heading   : 578


,filename,image_path,latitude,longitude,heading
0,Run 10_Camera 4 360_115_1.jpg,/Users/jiaweihu/Desktop/BBxB/Github_Version/Da...,39.303667,-76.583810,175.531865
1,Run 10_Camera 4 360_130_3.jpg,/Users/jiaweihu/Desktop/BBxB/Github_Version/Da...,39.294092,-76.654371,356.000542
2,Run 10_Camera 4 360_136_1.jpg,/Users/jiaweihu/Desktop/BBxB/Github_Version/Da...,39.294100,-76.654161,357.220324
3,Run 10_Camera 4 360_137_1.jpg,/Users/jiaweihu/Desktop/BBxB/Github_Version/Da...,39.294101,-76.654126,357.246696
4,Run 10_Camera 4 360_141_1.jpg,/Users/jiaweihu/Desktop/BBxB/Github_Version/Da...,39.294107,-76.653986,352.254918


## Geometry model

The camera field of view is approximated as a **triangular view cone** in a projected CRS.

That is a simplification, but it is:
- easy to explain,
- fast to compute,
- usually good enough for candidate generation.

In [4]:
def heading_to_math_angle(heading_deg):
    return (90 - float(heading_deg)) % 360

def point_from_xy_angle(x, y, angle_deg, dist):
    ang = math.radians(angle_deg)
    return (x + dist * math.cos(ang), y + dist * math.sin(ang))

def create_view_cone_projected(x, y, heading_deg, cone_length=CONE_LENGTH_FT, fov_deg=FOV_DEG):
    center_ang = heading_to_math_angle(heading_deg)
    half = fov_deg / 2.0
    left_pt  = point_from_xy_angle(x, y, center_ang + half, cone_length)
    right_pt = point_from_xy_angle(x, y, center_ang - half, cone_length)
    return Polygon([(x, y), left_pt, right_pt])

def bearing_from_points_projected(p1, p2):
    dx = p2.x - p1.x
    dy = p2.y - p1.y
    ang_math = math.degrees(math.atan2(dy, dx)) % 360
    return (90 - ang_math) % 360

def smallest_angle_diff_deg(a, b):
    diff = abs(float(a) - float(b)) % 360
    return min(diff, 360 - diff)

In [5]:
df_valid = df_imgs.dropna(subset=["latitude", "longitude", "heading"]).copy()

gdf_images = gpd.GeoDataFrame(
    df_valid,
    geometry=gpd.points_from_xy(df_valid["longitude"], df_valid["latitude"]),
    crs=CRS_WGS84,
).to_crs(CRS_PROJ)

gdf_images["camera_point"] = gdf_images.geometry.copy()
gdf_images["geometry"] = gdf_images.apply(
    lambda r: create_view_cone_projected(
        r["camera_point"].x,
        r["camera_point"].y,
        r["heading"],
        cone_length=CONE_LENGTH_FT,
        fov_deg=FOV_DEG,
    ),
    axis=1,
)

print("Image rows after valid GPS+heading filter:", len(gdf_images))
display(gdf_images[["filename", "latitude", "longitude", "heading"]].head())

Image rows after valid GPS+heading filter: 578


,filename,latitude,longitude,heading
0,Run 10_Camera 4 360_115_1.jpg,39.303667,-76.583810,175.531865
1,Run 10_Camera 4 360_130_3.jpg,39.294092,-76.654371,356.000542
2,Run 10_Camera 4 360_136_1.jpg,39.294100,-76.654161,357.220324
3,Run 10_Camera 4 360_137_1.jpg,39.294101,-76.654126,357.246696
4,Run 10_Camera 4 360_141_1.jpg,39.294107,-76.653986,352.254918


In [6]:
gdf_parcels = gpd.read_file(PARCEL_FILE)

if "BLOCKLOT" in gdf_parcels.columns and "property_id" not in gdf_parcels.columns:
    gdf_parcels = gdf_parcels.rename(columns={"BLOCKLOT": "property_id"})

gdf_parcels = gdf_parcels.to_crs(CRS_PROJ).copy()
gdf_parcels["parcel_geom"] = gdf_parcels.geometry
gdf_parcels["parcel_centroid_proj"] = gdf_parcels.geometry.centroid
gdf_parcels["parcel_rep_pt_proj"] = gdf_parcels.geometry.representative_point()

parcel_centroids_wgs84 = gpd.GeoSeries(
    gdf_parcels["parcel_centroid_proj"], crs=CRS_PROJ
).to_crs(CRS_WGS84)
gdf_parcels["parcel_latitude"] = parcel_centroids_wgs84.y
gdf_parcels["parcel_longitude"] = parcel_centroids_wgs84.x

# ---------------------------------------------------------
# Clean / standardize new Baltimore property-level fields
# Raw source columns in this GeoJSON:
#   SALEPRIC -> Property Sale Price
#   LDATE    -> Last Update (MMDDYYYY)
#   VACIND   -> Vacant Indicator (Y / N / missing)
# ---------------------------------------------------------

def parse_mmddyyyy_series(series):
    return pd.to_datetime(series.astype(str).str.strip(), format="%m%d%Y", errors="coerce")

if "SALEPRIC" in gdf_parcels.columns:
    gdf_parcels["property_sale_price"] = pd.to_numeric(gdf_parcels["SALEPRIC"], errors="coerce")
else:
    gdf_parcels["property_sale_price"] = np.nan

if "LDATE" in gdf_parcels.columns:
    gdf_parcels["last_update"] = parse_mmddyyyy_series(gdf_parcels["LDATE"])
    gdf_parcels["last_update_str"] = gdf_parcels["last_update"].dt.strftime("%Y-%m-%d")
else:
    gdf_parcels["last_update"] = pd.NaT
    gdf_parcels["last_update_str"] = pd.NA

if "VACIND" in gdf_parcels.columns:
    gdf_parcels["vacant_indicator"] = (
        gdf_parcels["VACIND"]
        .map({"Y": 1, "N": 0})
        .astype(float)
    )
else:
    gdf_parcels["vacant_indicator"] = np.nan

print("Parcel polygons loaded:", len(gdf_parcels))
display(
    gdf_parcels[
        [c for c in [
            "property_id", "FULLADDR", "parcel_latitude", "parcel_longitude",
            "SALEPRIC", "property_sale_price", "LDATE", "last_update_str",
            "VACIND", "vacant_indicator"
        ] if c in gdf_parcels.columns]
    ].head()
)

Parcel polygons loaded: 238491


,property_id,FULLADDR,parcel_latitude,parcel_longitude,SALEPRIC,property_sale_price,LDATE,last_update_str,VACIND,vacant_indicator
0,0001 001,2045 W NORTH AVE,39.309421,-76.651146,250000,250000,03082026,2026-03-08,None,NaN
1,0001 002,2043 W NORTH AVE,39.309424,-76.651093,250000,250000,03082026,2026-03-08,None,NaN
2,0001 003,2041 W NORTH AVE,39.309426,-76.651043,0,0,03082026,2026-03-08,Y,1.0
3,0001 004,2039 W NORTH AVE,39.309427,-76.650992,0,0,03082026,2026-03-08,Y,1.0
4,0001 005,2037 W NORTH AVE,39.309429,-76.650944,70000,70000,03082026,2026-03-08,None,NaN


In [7]:
candidate_cols_left = ["filename", "image_path", "latitude", "longitude", "heading", "camera_point", "geometry"]
candidate_cols_right = ["property_id", "FULLADDR", "geometry", "parcel_geom", "parcel_centroid_proj", "parcel_rep_pt_proj"]

candidates_all = gpd.sjoin(
    gdf_images[candidate_cols_left],
    gdf_parcels[candidate_cols_right],
    how="left",
    predicate="intersects",
).copy()

def compute_candidate_metrics(row):
    if pd.isna(row["property_id"]):
        return pd.Series(
            {
                "distance_boundary_ft": None,
                "distance_centroid_ft": None,
                "bearing_to_parcel_deg": None,
                "angle_diff_deg": None,
                "match_score": None,
            }
        )

    camera_pt = row["camera_point"]
    parcel_geom = row["parcel_geom"]
    parcel_centroid = row["parcel_centroid_proj"]
    parcel_rep_pt = row["parcel_rep_pt_proj"]

    distance_boundary = camera_pt.distance(parcel_geom)
    distance_centroid = camera_pt.distance(parcel_centroid)
    bearing = bearing_from_points_projected(camera_pt, parcel_rep_pt)
    angle_diff = smallest_angle_diff_deg(row["heading"], bearing)
    score = distance_boundary + ANGLE_WEIGHT_FT_PER_DEG * angle_diff

    return pd.Series(
        {
            "distance_boundary_ft": distance_boundary,
            "distance_centroid_ft": distance_centroid,
            "bearing_to_parcel_deg": bearing,
            "angle_diff_deg": angle_diff,
            "match_score": score,
        }
    )

candidates_all = pd.concat([candidates_all, candidates_all.apply(compute_candidate_metrics, axis=1)], axis=1)

print("Candidate rows:", len(candidates_all))
display(
    candidates_all[
        ["filename", "property_id", "FULLADDR", "distance_boundary_ft", "distance_centroid_ft", "angle_diff_deg", "match_score"]
    ].head(10)
)

Candidate rows: 3920


,filename,property_id,FULLADDR,distance_boundary_ft,distance_centroid_ft,angle_diff_deg,match_score
0,Run 10_Camera 4 360_115_1.jpg,1572 036,2333 E CHASE ST,101.826092,254.653143,6.972253,108.798345
0,Run 10_Camera 4 360_115_1.jpg,1572 001,2401 E CHASE ST,38.944792,74.358551,15.031039,53.975832
0,Run 10_Camera 4 360_115_1.jpg,1572 002,2403 E CHASE ST,37.500657,72.632345,4.771832,42.272489
0,Run 10_Camera 4 360_115_1.jpg,1572 003,2405 E CHASE ST,37.583335,72.972744,5.004073,42.587408
0,Run 10_Camera 4 360_115_1.jpg,1572 004,2407 E CHASE ST,40.273320,75.380865,14.162544,54.435864
0,Run 10_Camera 4 360_115_1.jpg,1572 005,2409 E CHASE ST,46.278235,79.703421,23.136368,69.414603
0,Run 10_Camera 4 360_115_1.jpg,1572 006,2411 E CHASE ST,54.556064,85.395816,30.500463,85.056528
0,Run 10_Camera 4 360_115_1.jpg,1572 007,2413 E CHASE ST,63.445617,92.501335,36.410574,99.856191
1,Run 10_Camera 4 360_130_3.jpg,2157 001,400 N SMALLWOOD ST,0.000000,183.839864,92.316274,92.316274
1,Run 10_Camera 4 360_130_3.jpg,2201 044,2340 LAURETTA AVE,32.831915,70.219628,15.832386,48.664301


In [8]:
candidates = candidates_all[
    candidates_all["property_id"].notna()
    & candidates_all["distance_boundary_ft"].notna()
    & (candidates_all["distance_boundary_ft"] <= MAX_BOUNDARY_DISTANCE_FT)
].copy()

print("Images total                     :", len(df_imgs))
print("Images with valid GPS+heading    :", gdf_images["filename"].nunique())
print("Images with >=1 parcel candidate :", candidates_all.loc[candidates_all["property_id"].notna(), "filename"].nunique())
print("Images after distance filter     :", candidates["filename"].nunique())
print()
print("Boundary distance stats:")
display(candidates["distance_boundary_ft"].describe())
print("Angle diff stats:")
display(candidates["angle_diff_deg"].describe())

Images total                     : 578
Images with valid GPS+heading    : 578
Images with >=1 parcel candidate : 578
Images after distance filter     : 578

Boundary distance stats:


count    3920.000000
mean       65.912762
std        24.674249
min         0.000000
25%        47.783987
50%        62.750163
75%        76.528435
max       129.172203
Name: distance_boundary_ft, dtype: float64

Angle diff stats:


count    3920.000000
mean       19.807169
std        13.126804
min         0.003565
25%         9.055564
50%        18.806442
75%        28.254810
max        98.698394
Name: angle_diff_deg, dtype: float64

In [9]:
# --- 12. Select one best parcel per image ---

candidates_valid = candidates.dropna(subset=["property_id"]).copy()

best_match = (
    candidates_valid
    .sort_values(["filename", "match_score", "distance_boundary_ft", "angle_diff_deg"])
    .groupby("filename", as_index=False)
    .first()
    .copy()
)

best_match = gpd.GeoDataFrame(best_match, geometry="geometry", crs=gdf_images.crs)

print("Best-matched images:", len(best_match))
print("Unique matched parcels:", best_match["property_id"].nunique())
display(
    best_match[
        ["filename", "property_id", "FULLADDR", "distance_boundary_ft", "angle_diff_deg", "match_score"]
    ].head(15)
)

Best-matched images: 578
Unique matched parcels: 502


,filename,property_id,FULLADDR,distance_boundary_ft,angle_diff_deg,match_score
0,Run 10_Camera 4 360_115_1.jpg,1572 002,2403 E CHASE ST,37.500657,4.771832,42.272489
1,Run 10_Camera 4 360_130_3.jpg,2201 045,2342 LAURETTA AVE,30.966650,4.655507,35.622157
2,Run 10_Camera 4 360_136_1.jpg,2201 041,2334 LAURETTA AVE,30.815083,1.062929,31.878012
3,Run 10_Camera 4 360_137_1.jpg,2201 040,2332 LAURETTA AVE,31.042462,2.181825,33.224287
4,Run 10_Camera 4 360_141_1.jpg,2201 038,2328 LAURETTA AVE,30.573625,3.411100,33.984725
5,Run 10_Camera 4 360_14_1.jpg,3801 014,1927 SAINT PAUL ST,34.090839,2.084824,36.175663
6,Run 10_Camera 4 360_151_1.jpg,2201 030,2312 LAURETTA AVE,30.996213,1.133748,32.129961
7,Run 10_Camera 4 360_152_1.jpg,2201 029,2310 LAURETTA AVE,31.323992,2.028102,33.352094
8,Run 10_Camera 4 360_169_1.jpg,2157 001,400 N SMALLWOOD ST,0.000000,20.572176,20.572176
9,Run 10_Camera 4 360_192_3.jpg,1572 037,2301 E CHASE ST,43.933759,49.378880,93.312638


In [10]:
# --- 13. Aggregate matched images to parcel level ---

parcel_image_agg = (
    best_match.groupby("property_id", as_index=False)
    .agg(
        image_names=("filename", lambda s: "|".join(sorted(set(map(str, s))))),
        image_paths=("image_path", lambda s: "|".join(sorted(set(map(str, s))))),
        image_count=("filename", "nunique"),
        matched_image_latitude_mean=("latitude", "mean"),
        matched_image_longitude_mean=("longitude", "mean"),
        min_boundary_distance_ft=("distance_boundary_ft", "min"),
        mean_boundary_distance_ft=("distance_boundary_ft", "mean"),
        min_angle_diff_deg=("angle_diff_deg", "min"),
        mean_angle_diff_deg=("angle_diff_deg", "mean"),
    )
)

parcel_all = gdf_parcels.merge(parcel_image_agg, on="property_id", how="left").copy()
parcel_all = gpd.GeoDataFrame(parcel_all, geometry="geometry", crs=gdf_parcels.crs)

parcel_matches = parcel_all.dropna(subset=["image_count"]).copy()
parcel_matches = gpd.GeoDataFrame(parcel_matches, geometry="geometry", crs=gdf_parcels.crs)

print("Unique parcels matched:", len(parcel_matches))
display(
    parcel_matches[
        ["property_id", "FULLADDR", "image_count", "image_names",
         "min_boundary_distance_ft", "mean_angle_diff_deg"]
    ].head(10)
)

Unique parcels matched: 505


,property_id,FULLADDR,image_count,image_names,min_boundary_distance_ft,mean_angle_diff_deg
3,0001 004,2039 W NORTH AVE,1.0,Run 18_Camera 4 360_276_3.jpg,56.806587,3.455000
8,0001 009,2029 W NORTH AVE,1.0,Run 18_Camera 4 360_268_1.jpg,56.979923,3.848565
9,0001 010,2027 W NORTH AVE,1.0,Run 18_Camera 4 360_267_1.jpg,57.064350,1.441362
15,0001 016,2001 W NORTH AVE,1.0,Run 18_Camera 4 360_256_3.jpg,57.889231,13.930158
68,0002 015,1901 W NORTH AVE,1.0,Run 18_Camera 4 360_199_1.jpg,55.365776,5.231363
71,0002 018,1907 W NORTH AVE,1.0,Run 18_Camera 4 360_204_3.jpg,56.142530,1.056664
80,0002 028,1929 W NORTH AVE,1.0,Run 18_Camera 4 360_223_3.jpg,57.484881,5.033667
84,0002 032,1937 W NORTH AVE,1.0,Run 18_Camera 4 360_230_3.jpg,57.461936,3.332913
134,0003 001,1800 N FULTON AVE,2.0,Run 23_Camera 4 360_140_1.jpg|Run 23_Camera 4 ...,72.793075,3.041846
135,0003 002,1802 N FULTON AVE,1.0,Run 23_Camera 4 360_143_3.jpg,72.821862,1.441202


## Manual feature labels → parcel-level dataframe

This section merges the uploaded manual annotation CSV onto the image-level matches, then aggregates those labels to the parcel level so `df_parcel` contains:

- unique property ID
- parcel centroid latitude / longitude
- associated image names and file paths
- one column per manually annotated feature
- optional counts showing how many matched images expressed each feature

Aggregation rule used below:
- binary parcel feature = **1 if any matched image for that parcel has the feature**
- feature count = number of matched matched images with that feature


In [11]:
# --- 14. Merge manual image labels and build feature-augmented df_parcel ---

# Keep the original/manual BCI file untouched and create a frontend-ready copy in Data/.
# The map UI uses a 0-10 BCI scale, so the generated CSV below is the file
# downstream cells should read. The guard prevents double-scaling if a source
# column has already been converted.
RAW_LABEL_FILE_CANDIDATES = [
    Path("../Front_End_BBxB/Data/Baltimore_StreetViewImages_Labels_Manual_withBCI.csv"),
    Path("Data/Baltimore_StreetViewImages_Labels_Manual_withBCI.csv"),
]
RAW_LABEL_FILE = next((p for p in RAW_LABEL_FILE_CANDIDATES if p.exists()), RAW_LABEL_FILE_CANDIDATES[-1])
LABEL_FILE = Path("Data/Baltimore_StreetViewImages_Labels_Manual_withBCI_0_10.csv")

BCI_SCALE_RANGES = {
    "BCI_1PL": (-1.1314697265625, 3.00828218460083),
    "BCI_2PL": (-1.6913913488388062, 1.4087016582489014),
}

def scale_bci_to_frontend(series, old_min, old_max):
    values = pd.to_numeric(series, errors="coerce")
    if values.dropna().between(0, 10).all() and values.max() > old_max:
        return values
    return (values - old_min) / (old_max - old_min) * 10

print(f"Raw BCI label source -> {RAW_LABEL_FILE.resolve()}")
raw_label_df = pd.read_csv(RAW_LABEL_FILE).copy()
for col, (old_min, old_max) in BCI_SCALE_RANGES.items():
    raw_label_df[col] = scale_bci_to_frontend(raw_label_df[col], old_min, old_max)

raw_label_df.to_csv(LABEL_FILE, index=False)
print(f"0-10 BCI label CSV saved -> {LABEL_FILE.resolve()}")

label_df = raw_label_df.copy()
label_df["filename"] = label_df["Filename"].astype(str).str.strip()

binary_feature_cols = [
    "Crooked_Roofline",
    "Broken_Steps",
    "Missing_Bricks",
    "Peeling_Paint",
    "Boarded_Windows",
    "Boarded_Doors",
    "Plant_Overgrowth",
    "Fire_Damage",
    "Cracked_Sidewalk",
    "Green_Space",
]

# Keep this alias for downstream cells that still expect `feature_cols`
feature_cols = binary_feature_cols.copy()

bci_cols = ["BCI_1PL", "BCI_2PL"]

# Binary manual labels -> numeric 0/1
for col in binary_feature_cols:
    label_df[col] = pd.to_numeric(label_df[col], errors="coerce").astype("Int64")

# BCI columns -> keep as float
for col in bci_cols:
    label_df[col] = pd.to_numeric(label_df[col], errors="coerce")

# Attach labels to image-level best matches
best_match_labeled = best_match.merge(
    label_df[["filename"] + binary_feature_cols + bci_cols],
    on="filename",
    how="left",
)

# Keep binary labels as nullable integers (do NOT fill missing with 0)
for col in binary_feature_cols:
    best_match_labeled[col] = pd.to_numeric(best_match_labeled[col], errors="coerce").astype("Int64")

# Keep BCI as float
for col in bci_cols:
    best_match_labeled[col] = pd.to_numeric(best_match_labeled[col], errors="coerce")

print("Image-level labeled matches:", len(best_match_labeled))
print(
    "Matched images with at least one positive binary label:",
    int((best_match_labeled[binary_feature_cols].sum(axis=1) > 0).sum())
)
print("Image-level non-null BCI_1PL:", int(best_match_labeled["BCI_1PL"].notna().sum()))
print("Image-level non-null BCI_2PL:", int(best_match_labeled["BCI_2PL"].notna().sum()))

display(best_match_labeled[["filename", "property_id"] + binary_feature_cols + bci_cols].head())


Raw BCI label source -> /Users/jiaweihu/Desktop/BBxB/Front_End_BBxB/Data/Baltimore_StreetViewImages_Labels_Manual_withBCI.csv
0-10 BCI label CSV saved -> /Users/jiaweihu/Desktop/BBxB/Github_Version/Data/Baltimore_StreetViewImages_Labels_Manual_withBCI_0_10.csv
Image-level labeled matches: 578
Matched images with at least one positive binary label: 235
Image-level non-null BCI_1PL: 289
Image-level non-null BCI_2PL: 289


,filename,property_id,Crooked_Roofline,Broken_Steps,Missing_Bricks,Peeling_Paint,Boarded_Windows,Boarded_Doors,Plant_Overgrowth,Fire_Damage,Cracked_Sidewalk,Green_Space,BCI_1PL,BCI_2PL
0,Run 10_Camera 4 360_115_1.jpg,1572 002,0,1,0,1,1,1,1,0,0,0,8.643640,7.213412
1,Run 10_Camera 4 360_130_3.jpg,2201 045,0,0,0,1,0,0,0,0,0,0,1.698058,2.427724
2,Run 10_Camera 4 360_136_1.jpg,2201 041,0,0,0,1,0,0,1,0,0,1,3.896164,3.195215
3,Run 10_Camera 4 360_137_1.jpg,2201 040,0,0,0,0,0,0,0,0,0,1,0.899202,1.226701
4,Run 10_Camera 4 360_141_1.jpg,2201 038,0,0,0,1,1,1,1,0,0,0,6.348050,5.881778


In [12]:
# --- 14b. Aggregate manual labels + BCI to parcel level ---

# Binary image labels -> parcel level
# Rule:
#   - parcel feature indicator = 1 if ANY matched image has the label
#   - parcel feature image count = number of matched images with that label
parcel_feature_any = (
    best_match_labeled.groupby("property_id", as_index=False)[binary_feature_cols]
    .max()
)

parcel_feature_counts = (
    best_match_labeled.groupby("property_id", as_index=False)[binary_feature_cols]
    .sum(min_count=1)
    .rename(columns={c: f"{c}_image_count" for c in binary_feature_cols})
)

# Continuous BCI -> parcel level
# Main parcel-level BCI is the MEAN across matched images, while min/max are
# retained for diagnostics / QA. We also keep a count of images that contributed
# non-null BCI values.
parcel_bci = (
    best_match_labeled.groupby("property_id", as_index=False)
    .agg(
        BCI_1PL=("BCI_1PL", "mean"),
        BCI_1PL_min=("BCI_1PL", "min"),
        BCI_1PL_max=("BCI_1PL", "max"),
        BCI_2PL=("BCI_2PL", "mean"),
        BCI_2PL_min=("BCI_2PL", "min"),
        BCI_2PL_max=("BCI_2PL", "max"),
        BCI_image_count=("BCI_1PL", lambda s: int(s.notna().sum()))
    )
)


# # Fill binary features only
# for col in binary_feature_cols:
#     df_parcel[col] = df_parcel[col].fillna(0).astype(int)
#     df_parcel[f"{col}_image_count"] = df_parcel[f"{col}_image_count"].fillna(0).astype(int)

df_parcel = (
    parcel_matches
    .merge(parcel_feature_any, on="property_id", how="left")
    .merge(parcel_feature_counts, on="property_id", how="left")
    .merge(parcel_bci, on="property_id", how="left")
    .copy()
)

# property_id that truly appear in the manual label file via matched images
labeled_property_ids = set(
    best_match_labeled.loc[
        best_match_labeled["filename"].isin(label_df["filename"]),
        "property_id"
    ].dropna().unique()
)

labeled_mask = df_parcel["property_id"].isin(labeled_property_ids)

# Only fill 0 for properties that were actually labeled.
# Unlabeled properties remain NA.
for col in binary_feature_cols:
    df_parcel.loc[labeled_mask, col] = df_parcel.loc[labeled_mask, col].fillna(0)
    df_parcel.loc[labeled_mask, f"{col}_image_count"] = (
        df_parcel.loc[labeled_mask, f"{col}_image_count"].fillna(0)
    )

    df_parcel[col] = df_parcel[col].astype("Int64")
    df_parcel[f"{col}_image_count"] = df_parcel[f"{col}_image_count"].astype("Int64")



    

# Keep BCI columns as float
bci_summary_cols = [
    "BCI_1PL", "BCI_1PL_min", "BCI_1PL_max",
    "BCI_2PL", "BCI_2PL_min", "BCI_2PL_max"
]
for col in bci_summary_cols:
    df_parcel[col] = pd.to_numeric(df_parcel[col], errors="coerce")

df_parcel["BCI_image_count"] = pd.to_numeric(
    df_parcel["BCI_image_count"], errors="coerce"
).astype("Int64")

# Keep geometry / CRS explicit
df_parcel = gpd.GeoDataFrame(df_parcel, geometry="geometry", crs=parcel_matches.crs)

print("df_parcel rows:", len(df_parcel))
print("Unique properties:", df_parcel["property_id"].nunique())
print("Non-null parcel BCI_1PL:", int(df_parcel["BCI_1PL"].notna().sum()))
print("Non-null parcel BCI_2PL:", int(df_parcel["BCI_2PL"].notna().sum()))

display_cols = [
    "property_id", "FULLADDR", "parcel_latitude", "parcel_longitude", "image_count",
    "property_sale_price", "last_update_str", "vacant_indicator",
    "BCI_1PL", "BCI_2PL", "BCI_image_count"
] + binary_feature_cols

display(df_parcel[[c for c in display_cols if c in df_parcel.columns]].head(10))


df_parcel rows: 505
Unique properties: 502
Non-null parcel BCI_1PL: 255
Non-null parcel BCI_2PL: 255


,property_id,FULLADDR,parcel_latitude,parcel_longitude,image_count,property_sale_price,last_update_str,vacant_indicator,BCI_1PL,BCI_2PL,...,Crooked_Roofline,Broken_Steps,Missing_Bricks,Peeling_Paint,Boarded_Windows,Boarded_Doors,Plant_Overgrowth,Fire_Damage,Cracked_Sidewalk,Green_Space
0,0001 004,2039 W NORTH AVE,39.309427,-76.650992,1.0,0,2026-03-08,1.0,NaN,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,0001 009,2029 W NORTH AVE,39.309435,-76.650748,1.0,0,2026-03-08,NaN,NaN,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,0001 010,2027 W NORTH AVE,39.309437,-76.650698,1.0,60000,2026-03-08,0.0,NaN,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,0001 016,2001 W NORTH AVE,39.309453,-76.650207,1.0,0,2026-03-08,NaN,NaN,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,0002 015,1901 W NORTH AVE,39.309544,-76.648289,1.0,43500,2026-03-08,1.0,NaN,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
5,0002 018,1907 W NORTH AVE,39.309536,-76.648440,1.0,0,2026-03-08,1.0,NaN,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
6,0002 028,1929 W NORTH AVE,39.309503,-76.649142,1.0,150000,2026-03-08,0.0,NaN,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
7,0002 032,1937 W NORTH AVE,39.309491,-76.649340,1.0,3000,2026-03-08,1.0,NaN,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
8,0003 001,1800 N FULTON AVE,39.308955,-76.646714,2.0,0,2026-03-08,1.0,NaN,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
9,0003 002,1802 N FULTON AVE,39.308997,-76.646719,1.0,20900,2026-03-08,NaN,NaN,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [13]:
# --- 15. Export feature-augmented parcel outputs (CSV / GeoJSON / Shapefile) ---


df_parcel_export = df_parcel.copy()

# ------------------------------------------------------------------
# 1) Preserve projected helper geometries as WKT for flat-file export
# ------------------------------------------------------------------
extra_geom_cols_parcel = ["parcel_geom", "parcel_centroid_proj", "parcel_rep_pt_proj"]
for col in extra_geom_cols_parcel:
    if col in df_parcel_export.columns:
        df_parcel_export[f"{col}_wkt"] = df_parcel_export[col].to_wkt()

# Create output folder

csv_path = output_dir / "df_parcel_feature_augmented.csv"
geojson_path = output_dir / "df_parcel_feature_augmented.geojson"
shp_base = "df_parcel_feature_augmented"
shp_path = output_dir / f"{shp_base}.shp"

# ------------------------------------------------------------------
# 2) GeoJSON export (WGS84)
# ------------------------------------------------------------------
df_parcel_geojson = df_parcel_export.drop(
    columns=[c for c in extra_geom_cols_parcel if c in df_parcel_export.columns],
    errors="ignore"
).copy()

df_parcel_geojson = gpd.GeoDataFrame(
    df_parcel_geojson,
    geometry="geometry",
    crs=df_parcel.crs
).to_crs(CRS_WGS84)

if geojson_path.exists():
    geojson_path.unlink()

df_parcel_geojson.to_file(geojson_path, driver="GeoJSON")

# ------------------------------------------------------------------
# 3) Shapefile export
#    - shorten field names
#    - keep only shapefile-safe columns
#    - remove old sidecar files before writing
# ------------------------------------------------------------------
df_parcel_shp = df_parcel_geojson.copy().rename(columns={
    "property_sale_price": "sale_price",
    "last_update_str": "last_upd",
    "vacant_indicator": "vacant_ind",
    "BCI_1PL": "bci_1pl",
    "BCI_2PL": "bci_2pl",
    "BCI_image_count": "bci_img_n",
})

shp_keep_cols = [
    "property_id",
    "FULLADDR",
    "image_count",
    "sale_price",
    "last_upd",
    "vacant_ind",
    "bci_1pl",
    "bci_2pl",
    "bci_img_n",
    "parcel_latitude",
    "parcel_longitude",
    "geometry",
]

df_parcel_shp = df_parcel_shp[
    [c for c in shp_keep_cols if c in df_parcel_shp.columns]
].copy()

for ext in [".shp", ".shx", ".dbf", ".prj", ".cpg"]:
    f = output_dir / f"{shp_base}{ext}"
    if f.exists():
        try:
            f.unlink()
        except PermissionError:
            print(f"Warning: could not delete {f}; it may be open in another program.")

df_parcel_shp.to_file(shp_path, driver="ESRI Shapefile")

# ------------------------------------------------------------------
# 4) CSV export
# ------------------------------------------------------------------
df_parcel_csv = df_parcel_export.copy()
df_parcel_csv["geometry_wkt"] = df_parcel_csv.geometry.to_wkt()
df_parcel_csv = pd.DataFrame(df_parcel_csv.drop(columns="geometry"))

if csv_path.exists():
    csv_path.unlink()

df_parcel_csv.to_csv(csv_path, index=False)

# Frontend-ready CSV copy in Data/ with BCI_1PL already on the 0-10 scale.
data_csv_path = Path("Data/df_parcel_feature_augmented_0_10.csv")
df_parcel_csv.to_csv(data_csv_path, index=False)

print(f"CSV saved     -> {csv_path.resolve()}")
print(f"Data CSV saved -> {data_csv_path.resolve()}")
print(f"GeoJSON saved -> {geojson_path.resolve()}")
print(f"Shapefile saved -> {shp_path.resolve()}")
print("CSV now includes parcel-level BCI_1PL / BCI_2PL plus min/max diagnostics.")
print("BCI_1PL is on the frontend 0-10 scale in the Data CSV.")


CSV saved     -> /Users/jiaweihu/Desktop/BBxB/Github_Version/outputs/df_parcel_feature_augmented.csv
Data CSV saved -> /Users/jiaweihu/Desktop/BBxB/Github_Version/Data/df_parcel_feature_augmented_0_10.csv
GeoJSON saved -> /Users/jiaweihu/Desktop/BBxB/Github_Version/outputs/df_parcel_feature_augmented.geojson
Shapefile saved -> /Users/jiaweihu/Desktop/BBxB/Github_Version/outputs/df_parcel_feature_augmented.shp
CSV now includes parcel-level BCI_1PL / BCI_2PL plus min/max diagnostics.
BCI_1PL is on the frontend 0-10 scale in the Data CSV.


/var/folders/y_/0hljknn13vb1y77zbd8j0lq80000gn/T/ipykernel_6374/2008102439.py:82: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  df_parcel_shp.to_file(shp_path, driver="ESRI Shapefile")
/Users/jiaweihu/miniconda3/envs/jupyterlab/lib/python3.10/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'property_id' to 'property_i'
  ogr_write(
/Users/jiaweihu/miniconda3/envs/jupyterlab/lib/python3.10/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'image_count' to 'image_coun'
  ogr_write(
/Users/jiaweihu/miniconda3/envs/jupyterlab/lib/python3.10/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'parcel_latitude' to 'parcel_lat'
  ogr_write(
/Users/jiaweihu/miniconda3/envs/jupyterlab/lib/python3.10/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'parcel_longitude' to 'parcel_lon'
  ogr_write(


In [14]:
# --- 16. Single Folium map with:
#     - vacant / not vacant / all filter
#     - optional building condition filter
#     - optional numeric value filter
#     - top-right legend for Property Sale Price / BCI 1 / BCI 2
#     - NO separate "Symbol Color" block
#     - save output into outputs/

import json
from pathlib import Path
from branca.element import Element

# ------------------------------------------------------------
# Prepare data
# ------------------------------------------------------------
df_parcel_wgs84 = df_parcel.to_crs("EPSG:4326").copy()


m_features = folium.Map(
    location=[
        df_parcel_wgs84["parcel_latitude"].mean(),
        df_parcel_wgs84["parcel_longitude"].mean()
    ],
    zoom_start=16,
    tiles="CartoDB positron",
    zoom_control=False
)

map_var = m_features.get_name()

# Put zoom back on top-right, but wait until folium map exists
zoom_js = f"""
<script>
(function() {{
  function addZoomWhenReady() {{
    var mapObj = window["{map_var}"];
    if (mapObj) {{
      L.control.zoom({{ position: 'topright' }}).addTo(mapObj);
    }} else {{
      setTimeout(addZoomWhenReady, 50);
    }}
  }}
  if (document.readyState === "loading") {{
    document.addEventListener("DOMContentLoaded", addZoomWhenReady);
  }} else {{
    addZoomWhenReady();
  }}
}})();
</script>
"""
m_features.get_root().html.add_child(Element(zoom_js))


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
CONDITION_COLUMNS = [
    ("Crooked Roofline", "Crooked_Roofline"),
    ("Broken Steps", "Broken_Steps"),
    ("Missing Bricks", "Missing_Bricks"),
    ("Peeling Paint", "Peeling_Paint"),
    ("Boarded Windows", "Boarded_Windows"),
    ("Boarded Doors", "Boarded_Doors"),
    ("Plant Overgrowth", "Plant_Overgrowth"),
    ("Fire Damage", "Fire_Damage"),
]

COLOR_METRICS = [
    ("Property Sale Price", "property_sale_price"),
    ("BCI 1PL", "BCI_1PL"),
    ("BCI 2PL", "BCI_2PL"),
]

from urllib.parse import quote

# GitHub Pages serves index.html and testImages/ from the same repo folder.
# Use relative web paths, not file:///Users/... paths, so images load after deploy.
IMAGE_WEB_PREFIX = "testImages/"

def path_to_image_src(path_str):
    try:
        if path_str is None or pd.isna(path_str):
            return None

        filename = Path(str(path_str)).name
        return IMAGE_WEB_PREFIX + quote(filename)
    except Exception:
        return None
def format_price(val):
    if val is None or pd.isna(val):
        return "NA"
    return f"${float(val):,.0f}"

def format_metric(val):
    if val is None or pd.isna(val):
        return "NA"
    return f"{float(val):.2f}"

def make_popup(row):
    image_html = ""
    image_paths = str(row.get("image_paths", "")).split("|") if pd.notna(row.get("image_paths")) else []
    image_names = str(row.get("image_names", "")).split("|") if pd.notna(row.get("image_names")) else []

    if len(image_paths) > 0 and str(image_paths[0]).strip():
        img_uri = path_to_image_src(image_paths[0])
        img_name = image_names[0] if len(image_names) > 0 else Path(image_paths[0]).name
        if img_uri:
            image_html = f"""
            <div style="margin-top:10px;">
                <b>{escape(str(img_name))}</b><br>
                <img src="{img_uri}" style="width:240px; max-height:180px; object-fit:contain; border:1px solid #ccc; margin-top:4px;">
            </div>
            """

    vacant_raw = row.get("VACIND")
    if pd.isna(vacant_raw):
        vacant_txt = "NA"
    elif str(vacant_raw).strip().upper() == "Y":
        vacant_txt = "Vacant"
    else:
        vacant_txt = "Not vacant"

    condition_list = []
    for label, col in CONDITION_COLUMNS:
        val = row.get(col)
        if not pd.isna(val) and int(val) == 1:
            condition_list.append(label)
    condition_txt = ", ".join(condition_list) if condition_list else "None"

    return f"""
    <div style="font-family:sans-serif; font-size:13px; line-height:1.35;">
      <div style="font-size:15px; font-weight:bold; margin-bottom:6px;">Property Detail</div>
      <b>Property ID:</b> {escape(str(row.get("property_id", "NA")))}<br>
      <b>Address:</b> {escape(str(row.get("FULLADDR", "NA")))}<br>
      <b>Latitude:</b> {row.get("parcel_latitude", "NA")}<br>
      <b>Longitude:</b> {row.get("parcel_longitude", "NA")}<br>
      <b>Sale Price:</b> {format_price(row.get("property_sale_price"))}<br>
      <b>BCI 1PL:</b> {format_metric(row.get("BCI_1PL"))}<br>
      <b>BCI 2PL:</b> {format_metric(row.get("BCI_2PL"))}<br>
      <b>Vacancy:</b> {vacant_txt}<br>
      <b>Condition Features:</b> {escape(condition_txt)}
      {image_html}
    </div>
    """


# ------------------------------------------------------------
# Build JS records
# ------------------------------------------------------------
records = []

for _, row in df_parcel_wgs84.iterrows():
    lat = row.get("parcel_latitude")
    lon = row.get("parcel_longitude")

    if pd.isna(lat) or pd.isna(lon):
        continue

    vacant_raw = row.get("VACIND")
    if pd.isna(vacant_raw):
        vacant_status = "unknown"
    elif str(vacant_raw).strip().upper() == "Y":
        vacant_status = "vacant"
    else:
        vacant_status = "not_vacant"

    cond_dict = {}
    for label, col in CONDITION_COLUMNS:
        val = row.get(col)
        cond_dict[col] = 0 if pd.isna(val) else int(val)

    records.append({
        "lat": float(lat),
        "lon": float(lon),
        "popup": make_popup(row),
        "vacant_status": vacant_status,
        "property_sale_price": None if pd.isna(row.get("property_sale_price")) else float(row.get("property_sale_price")),
        "BCI_1PL": None if pd.isna(row.get("BCI_1PL")) else float(row.get("BCI_1PL")),
        "BCI_2PL": None if pd.isna(row.get("BCI_2PL")) else float(row.get("BCI_2PL")),
        "conditions": cond_dict
    })

records_json = json.dumps(records)
condition_json = json.dumps(CONDITION_COLUMNS)
color_metric_json = json.dumps(COLOR_METRICS)


# ------------------------------------------------------------
# HTML / CSS / JS
# ------------------------------------------------------------
control_html = f"""
<style>
#filter-toggle {{
  position: fixed;
  top: 14px;
  left: 14px;
  z-index: 10001;
  width: 36px;
  height: 36px;
  border-radius: 10px;
  border: 1px solid #cfcfcf;
  background: white;
  box-shadow: 0 2px 8px rgba(0,0,0,0.15);
  cursor: pointer;
  font-size: 22px;
  line-height: 34px;
  text-align: center;
  font-family: sans-serif;
}}

#filter-panel {{
  position: fixed;
  top: 56px;
  left: 14px;
  z-index: 10000;
  width: 320px;
  max-height: calc(100vh - 80px);
  overflow-y: auto;
  background: rgba(255,255,255,0.96);
  border: 1px solid #d0d0d0;
  border-radius: 16px;
  box-shadow: 0 4px 14px rgba(0,0,0,0.16);
  padding: 14px 14px 16px 14px;
  font-family: sans-serif;
}}

#filter-panel.hidden {{
  display: none;
}}

#panel-header {{
  display: flex;
  justify-content: space-between;
  align-items: flex-start;
  margin-bottom: 10px;
}}

#panel-title {{
  font-size: 16px;
  font-weight: 700;
  line-height: 1.1;
}}

#panel-subtitle {{
  font-size: 11px;
  color: #555;
  margin-top: 4px;
}}

#panel-close {{
  border: none;
  background: #efefef;
  width: 28px;
  height: 28px;
  border-radius: 8px;
  cursor: pointer;
  font-size: 18px;
}}

.section {{
  margin-top: 12px;
  padding-top: 10px;
  border-top: 1px solid #e7e7e7;
}}

.section-title {{
  font-weight: 700;
  font-size: 13px;
  margin-bottom: 6px;
}}

.section-note {{
  font-size: 11px;
  color: #666;
  margin-bottom: 6px;
}}

.select-full, .number-input {{
  width: 100%;
  box-sizing: border-box;
  padding: 5px 6px;
  border: 1px solid #c9c9c9;
  border-radius: 6px;
  background: white;
}}

.two-col {{
  display: grid;
  grid-template-columns: 1fr 1fr;
  gap: 8px;
}}

.btn-row {{
  display: flex;
  gap: 8px;
  flex-wrap: wrap;
  margin-top: 8px;
}}

.btn {{
  padding: 6px 12px;
  border-radius: 10px;
  border: 1px solid #cfcfcf;
  background: #f7f7f7;
  cursor: pointer;
  font-size: 12px;
}}

.btn-primary {{
  background: #2f80ed;
  color: white;
  border: 1px solid #2f80ed;
}}

.condition-grid {{
  display: grid;
  grid-template-columns: 1fr 1fr;
  gap: 6px 12px;
  margin-top: 8px;
  padding: 10px;
  border: 1px solid #d8d8d8;
  border-radius: 10px;
  background: #fafafa;
}}

.condition-item {{
  display: flex;
  align-items: center;
  gap: 6px;
  font-size: 12px;
}}

.radio-row {{
  display: flex;
  gap: 14px;
  align-items: center;
  margin-top: 8px;
  font-size: 12px;
}}

.summary-box {{
  margin-top: 12px;
  padding: 10px;
  border-radius: 10px;
  background: #f3f3f3;
  font-size: 12px;
  color: #333;
}}

#legend-box {{
  position: fixed;
  top: 14px;
  right: 70px;
  z-index: 10000;
  background: white;
  padding: 10px 12px;
  border-radius: 10px;
  border: 1px solid #aaa;
  box-shadow: 0 4px 10px rgba(0,0,0,0.18);
  font-family: sans-serif;
  min-width: 220px;
  display: none;
}}

#legend-title {{
  font-weight: bold;
  font-size: 14px;
  margin-bottom: 8px;
}}

#legend-gradient {{
  height: 14px;
  border-radius: 8px;
  border: 1px solid #ccc;
  background: linear-gradient(to right, #2ca02c, #f1e05a, #ef3b2c);
  margin-bottom: 6px;
}}

#legend-labels {{
  display: flex;
  justify-content: space-between;
  font-size: 11px;
  color: #333;
}}
</style>

<div id="filter-toggle" onclick="togglePanel()">☰</div>

<div id="filter-panel">
  <div id="panel-header">
    <div>
      <div id="panel-title">Map Filters</div>
      <div id="panel-subtitle">Filter parcel points. Color legend is shown at top right.</div>
    </div>
    <button id="panel-close" onclick="togglePanel()">×</button>
  </div>

  <div class="section" style="margin-top:0; padding-top:0; border-top:none;">
    <div class="section-title">1) Property status</div>
    <div style="font-weight:600; font-size:12px; margin-bottom:4px;">Show parcels</div>
    <select id="status_select" class="select-full">
      <option value="all">All parcels</option>
      <option value="vacant">Vacant only</option>
      <option value="not_vacant">Not vacant only</option>
    </select>
  </div>

  <div class="section">
    <div class="section-title">2) Building condition</div>
    <div class="section-note">Select one or more conditions to filter the visible parcels.</div>

    <div class="btn-row">
      <button class="btn" onclick="selectAllConditions()">Select all</button>
      <button class="btn" onclick="clearAllConditions()">Clear all</button>
    </div>

    <div class="condition-grid" id="condition_grid"></div>

    <div class="radio-row">
      <label><input type="radio" name="condition_mode" value="any" checked> Match any selected</label>
      <label><input type="radio" name="condition_mode" value="all"> Match all selected</label>
    </div>
  </div>

  <div class="section">
    <div class="section-title">3) Value filter</div>
    <label style="font-size:12px;">
      <input type="checkbox" id="enable_value_filter">
      Enable numeric filter
    </label>

    <div style="margin-top:8px; font-weight:600; font-size:12px;">Metric</div>
    <select id="value_metric_select" class="select-full" style="margin-top:4px;">
      <option value="property_sale_price">Property Sale Price</option>
      <option value="BCI_1PL">BCI 1PL</option>
      <option value="BCI_2PL">BCI 2PL</option>
    </select>

    <div class="two-col" style="margin-top:8px;">
      <div>
        <div style="font-size:12px; font-weight:600; margin-bottom:4px;">Min</div>
        <input type="number" id="min_value_input" class="number-input">
      </div>
      <div>
        <div style="font-size:12px; font-weight:600; margin-bottom:4px;">Max</div>
        <input type="number" id="max_value_input" class="number-input">
      </div>
    </div>

    <div class="btn-row">
      <button class="btn" onclick="resetValueRange()">Reset range</button>
    </div>
  </div>

  <div class="btn-row" style="margin-top:14px;">
    <button class="btn btn-primary" onclick="renderParcels()">Apply</button>
    <button class="btn" onclick="resetAllFilters()">Reset all</button>
  </div>

  <div class="summary-box" id="summary_box"></div>
</div>

<div id="legend-box">
  <div id="legend-title">Legend</div>
  <div id="legend-gradient"></div>
  <div id="legend-labels">
    <span id="legend-min"></span>
    <span id="legend-max"></span>
  </div>
</div>

<script>
var parcelData = {records_json};
var conditionColumns = {condition_json};
var colorMetrics = {color_metric_json};

var map = null;
var parcelLayer = null;
var activeColorMetric = "property_sale_price";

function togglePanel() {{
  var panel = document.getElementById("filter-panel");
  panel.classList.toggle("hidden");
}}

function buildConditionCheckboxes() {{
  var grid = document.getElementById("condition_grid");
  grid.innerHTML = "";
  conditionColumns.forEach(function(item) {{
    var label = item[0];
    var col = item[1];
    var div = document.createElement("label");
    div.className = "condition-item";
    div.innerHTML = `<input type="checkbox" class="condition-checkbox" value="${{col}}"> ${{label}}`;
    grid.appendChild(div);
  }});
}}

function getSelectedConditions() {{
  return Array.from(document.querySelectorAll(".condition-checkbox:checked")).map(x => x.value);
}}

function selectAllConditions() {{
  document.querySelectorAll(".condition-checkbox").forEach(x => x.checked = true);
}}

function clearAllConditions() {{
  document.querySelectorAll(".condition-checkbox").forEach(x => x.checked = false);
}}

function getConditionMode() {{
  var el = document.querySelector('input[name="condition_mode"]:checked');
  return el ? el.value : "any";
}}

function getMetricValues(metricName) {{
  return parcelData
    .map(d => d[metricName])
    .filter(v => v !== null && v !== undefined && !isNaN(v) && isFinite(v));
}}

function metricPrettyName(metricName) {{
  if (metricName === "property_sale_price") return "Property Sale Price";
  if (metricName === "BCI_1PL") return "BCI 1PL";
  if (metricName === "BCI_2PL") return "BCI 2PL";
  return metricName;
}}

function formatLegendValue(metricName, value) {{
  if (metricName === "property_sale_price") {{
    return "$" + Math.round(value).toLocaleString();
  }}
  return Number(value).toFixed(2);
}}

function updateLegend(metricName, minVal, maxVal) {{
  var box = document.getElementById("legend-box");
  if (minVal === null || maxVal === null) {{
    box.style.display = "none";
    return;
  }}
  document.getElementById("legend-title").innerText = metricPrettyName(metricName);
  document.getElementById("legend-min").innerText = formatLegendValue(metricName, minVal);
  document.getElementById("legend-max").innerText = formatLegendValue(metricName, maxVal);
  box.style.display = "block";
}}

function lerp(a, b, t) {{
  return a + (b - a) * t;
}}

function getColorFromRamp(value, minVal, maxVal) {{
  if (value === null || value === undefined || isNaN(value) || !isFinite(value)) {{
    return "#bdbdbd";
  }}
  if (maxVal <= minVal) {{
    return "rgb(241,224,90)";
  }}

  var t = (value - minVal) / (maxVal - minVal);
  t = Math.max(0, Math.min(1, t));

  var r, g, b;

  if (t < 0.5) {{
    var tt = t / 0.5;
    r = Math.round(lerp(44, 241, tt));
    g = Math.round(lerp(160, 224, tt));
    b = Math.round(lerp(44, 90, tt));
  }} else {{
    var tt = (t - 0.5) / 0.5;
    r = Math.round(lerp(241, 239, tt));
    g = Math.round(lerp(224, 59, tt));
    b = Math.round(lerp(90, 44, tt));
  }}

  return `rgb(${{r}}, ${{g}}, ${{b}})`;
}}

function parcelPassesStatus(d, statusMode) {{
  if (statusMode === "all") return true;
  return d.vacant_status === statusMode;
}}

function parcelPassesConditions(d, selectedConditions, mode) {{
  if (selectedConditions.length === 0) return true;
  var hits = selectedConditions.map(c => (d.conditions[c] || 0) === 1);
  if (mode === "all") return hits.every(v => v);
  return hits.some(v => v);
}}

function parcelPassesValueFilter(d, enabled, metricName, minVal, maxVal) {{
  if (!enabled) return true;
  var v = d[metricName];
  if (v === null || v === undefined || isNaN(v) || !isFinite(v)) return false;
  return v >= minVal && v <= maxVal;
}}


function resetValueRange() {{
  var metric = document.getElementById("value_metric_select").value;
  var vals = getMetricValues(metric);
  if (vals.length === 0) {{
    document.getElementById("min_value_input").value = "";
    document.getElementById("max_value_input").value = "";
    return;
  }}
  document.getElementById("min_value_input").value = Math.min(...vals).toFixed(2);
  document.getElementById("max_value_input").value = Math.max(...vals).toFixed(2);
}}

function resetAllFilters() {{
  document.getElementById("status_select").value = "all";
  document.getElementById("enable_value_filter").checked = false;
  document.getElementById("value_metric_select").value = "property_sale_price";
  clearAllConditions();
  document.querySelector('input[name="condition_mode"][value="any"]').checked = true;
  resetValueRange();
  activeColorMetric = "property_sale_price";
  renderParcels();
}}

function renderParcels() {{
  if (!parcelLayer) return;

  parcelLayer.clearLayers();

  var statusMode = document.getElementById("status_select").value;
  var selectedConditions = getSelectedConditions();
  var conditionMode = getConditionMode();

  var valueEnabled = document.getElementById("enable_value_filter").checked;
  var valueMetric = document.getElementById("value_metric_select").value;
  var minFilter = parseFloat(document.getElementById("min_value_input").value);
  var maxFilter = parseFloat(document.getElementById("max_value_input").value);

// Make legend + point coloring follow the selected numeric metric
activeColorMetric = valueMetric;

  if (isNaN(minFilter) || isNaN(maxFilter)) {{
    var vals0 = getMetricValues(valueMetric);
    if (vals0.length > 0) {{
      minFilter = Math.min(...vals0);
      maxFilter = Math.max(...vals0);
    }}
  }}

  var filtered = parcelData.filter(function(d) {{
    return parcelPassesStatus(d, statusMode) &&
           parcelPassesConditions(d, selectedConditions, conditionMode) &&
           parcelPassesValueFilter(d, valueEnabled, valueMetric, minFilter, maxFilter);
  }});

    var colorMin = null;
    var colorMax = null;
    
    if (activeColorMetric === "BCI_1PL" || activeColorMetric === "BCI_2PL") {{
      colorMin = 0;
      colorMax = 10;
    }} else {{
      var colorVals = filtered
        .map(d => d[activeColorMetric])
        .filter(v => v !== null && v !== undefined && !isNaN(v) && isFinite(v));
    
      if (colorVals.length > 0) {{
        colorMin = Math.min(...colorVals);
        colorMax = Math.max(...colorVals);
      }}
    }}

  filtered.forEach(function(d) {{
    var fill = "#3b6fc4";
    if (colorMin !== null && colorMax !== null) {{
      fill = getColorFromRamp(d[activeColorMetric], colorMin, colorMax);
    }}

    L.circleMarker([d.lat, d.lon], {{
      radius: 5,
      color: fill,
      weight: 1,
      fillColor: fill,
      fillOpacity: 0.9
    }})
    .bindPopup(d.popup, {{maxWidth: 320}})
    .addTo(parcelLayer);
  }});

  updateLegend(activeColorMetric, colorMin, colorMax);

  var statusText = "All parcels";
  if (statusMode === "vacant") statusText = "Vacant only";
  if (statusMode === "not_vacant") statusText = "Not vacant only";

  var condText = selectedConditions.length === 0
    ? "All conditions"
    : selectedConditions.length + " selected";

  var condModeText = selectedConditions.length === 0
    ? "-"
    : conditionMode.toUpperCase();

  var valueText = valueEnabled
    ? metricPrettyName(valueMetric) + ": " + formatLegendValue(valueMetric, minFilter) + " to " + formatLegendValue(valueMetric, maxFilter)
    : "Off";

  document.getElementById("summary_box").innerHTML =
    "<b>" + filtered.length.toLocaleString() + " parcels visible</b><br>" +
    "Status: " + statusText + "<br>" +
    "Condition: " + condText + "<br>" +
    "Condition mode: " + condModeText + "<br>" +
    "Value filter: " + valueText + "<br>" +
    "Color legend: " + metricPrettyName(activeColorMetric);
}}

function initializeFrontend() {{
  map = window["{map_var}"];
  if (!map) {{
    console.error("Folium map object not ready yet.");
    return;
  }}

  parcelLayer = L.layerGroup().addTo(map);
  buildConditionCheckboxes();
  resetValueRange();
  renderParcels();
}}

function waitForMapAndInit() {{
  var tries = 0;
  var maxTries = 200;

  function check() {{
    if (window["{map_var}"]) {{
      initializeFrontend();
      return;
    }}
    tries += 1;
    if (tries < maxTries) {{
      setTimeout(check, 50);
    }} else {{
      console.error("Timed out waiting for Folium map object: {map_var}");
    }}
  }}

  check();
}}

if (document.readyState === "loading") {{
  document.addEventListener("DOMContentLoaded", function() {{
    waitForMapAndInit();
  }});
}} else {{
  waitForMapAndInit();
}}

document.addEventListener("change", function(e) {{
  if (e.target && e.target.id === "value_metric_select") {{
    resetValueRange();
    renderParcels();
  }}
}});
</script>
"""

m_features.get_root().html.add_child(Element(control_html))

output_html = output_dir / "parcel_map_with_price_bci_colormap.html"
m_features.save(str(output_html))
print(f"Saved: {output_html}")

Saved: outputs/parcel_map_with_price_bci_colormap.html


In [15]:
# --- Diagnostics: verify all best-matched images are represented in parcel_matches ---

parcel_match_imgs = set()
for s in parcel_matches["image_names"].dropna():
    for img in str(s).split("|"):
        parcel_match_imgs.add(img.strip())

missing_imgs = set(best_match["filename"]) - parcel_match_imgs
extra_imgs = parcel_match_imgs - set(best_match["filename"])

print("Images in best_match but not represented in parcel_matches:", missing_imgs)
print("Images represented in parcel_matches but not in best_match:", extra_imgs)
print("Count in best_match:", best_match["filename"].nunique())
print("Count represented through parcel_matches:", len(parcel_match_imgs))

Images in best_match but not represented in parcel_matches: set()
Images represented in parcel_matches but not in best_match: set()
Count in best_match: 578
Count represented through parcel_matches: 578


In [16]:
# --- Export outputs ---

# ========= 1) image-level results =========
best_match_export = best_match.copy()

extra_geom_cols_best = ["camera_point", "parcel_geom", "parcel_centroid_proj", "parcel_rep_pt_proj"]
for col in extra_geom_cols_best:
    if col in best_match_export.columns:
        best_match_export[f"{col}_wkt"] = best_match_export[col].to_wkt()

best_match_geojson = best_match_export.drop(
    columns=[c for c in extra_geom_cols_best if c in best_match_export.columns],
    errors="ignore"
).copy()

best_match_geojson = gpd.GeoDataFrame(
    best_match_geojson,
    geometry="geometry",
    crs=best_match.crs
).to_crs("EPSG:4326")

best_match_geojson.to_file("best_match.geojson", driver="GeoJSON")

best_match_csv = best_match_export.copy()
best_match_csv["geometry_wkt"] = best_match_csv.geometry.to_wkt()
best_match_csv = pd.DataFrame(best_match_csv.drop(columns="geometry"))
best_match_csv.to_csv("best_match.csv", index=False)

# ========= 2) parcel-level matched results =========
parcel_matches_export = parcel_matches.copy()

extra_geom_cols_parcel = ["parcel_geom", "parcel_centroid_proj", "parcel_rep_pt_proj"]
for col in extra_geom_cols_parcel:
    if col in parcel_matches_export.columns:
        parcel_matches_export[f"{col}_wkt"] = parcel_matches_export[col].to_wkt()

parcel_matches_geojson = parcel_matches_export.drop(
    columns=[c for c in extra_geom_cols_parcel if c in parcel_matches_export.columns],
    errors="ignore"
).copy()

parcel_matches_geojson = gpd.GeoDataFrame(
    parcel_matches_geojson,
    geometry="geometry",
    crs=parcel_matches.crs
).to_crs("EPSG:4326")

parcel_matches_geojson.to_file("parcel_matches.geojson", driver="GeoJSON")

parcel_matches_csv = parcel_matches_export.copy()
parcel_matches_csv["geometry_wkt"] = parcel_matches_csv.geometry.to_wkt()
parcel_matches_csv = pd.DataFrame(parcel_matches_csv.drop(columns="geometry"))
parcel_matches_csv.to_csv("parcel_matches.csv", index=False)

print("Saved -> best_match.geojson")
print("Saved -> best_match.csv")
print("Saved -> parcel_matches.geojson")
print("Saved -> parcel_matches.csv")

Saved -> best_match.geojson
Saved -> best_match.csv
Saved -> parcel_matches.geojson
Saved -> parcel_matches.csv


In [17]:
images_with_candidates = set(candidates_all.loc[candidates_all["property_id"].notna(), "filename"].unique())
images_after_filter = set(candidates["filename"].unique())
best_match_images = set(best_match["filename"].unique())

dropped_by_distance = sorted(images_with_candidates - images_after_filter)
missing_after_best_match = sorted(images_after_filter - best_match_images)

parcel_matched_rows = parcel_matches.loc[parcel_matches["image_count"].notna()].copy()

images_shown_on_parcel_layer = set()
for s in parcel_matched_rows["image_names"].dropna():
    for img in str(s).split("|"):
        img = img.strip()
        if img:
            images_shown_on_parcel_layer.add(img)

not_in_parcel_aggregation = sorted(best_match_images - images_shown_on_parcel_layer)

print("Images dropped by distance filter:", len(dropped_by_distance))
print(dropped_by_distance)
print()
print("Images missing after best_match step:", len(missing_after_best_match))
print(missing_after_best_match)
print()
print("Best-match images not represented in parcel aggregation:", len(not_in_parcel_aggregation))
print(not_in_parcel_aggregation)
print()
print("best_match image count:", len(best_match_images))
print("parcel marker count   :", parcel_matched_rows.shape[0])

Images dropped by distance filter: 0
[]

Images missing after best_match step: 0
[]

Best-match images not represented in parcel aggregation: 0
[]

best_match image count: 578
parcel marker count   : 505
